## Завдання 1: Передбачення ймовірності кредитного дефолту
### Опис:
- Банк хоче передбачити, чи клієнт може не повернути кредит (default) на основі історії транзакцій та демографічних даних.
- Дані:
    - Числові ознаки:
        - credit_amount – сума кредиту
        - income – річний дохід клієнта
        - age – вік клієнта
        - loan_duration – тривалість кредиту в місяцях
    - Категоріальні ознаки:
        - gender – стать
        - marital_status – сімейний стан
        - job_type – тип роботи
    - Цільова змінна:
        - default (0 = повернув кредит, 1 = не повернув)
- Завдання студента:
1. Обробити дані:
    - Перетворити категоріальні ознаки у числові (даммі-ознаки).
    - Масштабувати числові ознаки.
2. Розбити дані на тренувальний та тестовий набори.
3. Створити нейронну мережу для бінарної класифікації:
    - 2–3 приховані шари з активацією ReLU
    - Вихідний шар з Sigmoid
4. Визначити функцію втрат (BCELoss) (Це лічильник помилок. Якщо комп'ютер помилився, цей лічильник показує велике число. Якщо він відгадав правильно, число маленьке.) та оптимізатор (наприклад, Adam) (Це "вчитель", який допомагає комп'ютеру вчитися. 
   Він каже комп'ютеру: "ти помилився, спробуй змінити ці числа ось так, щоб наступного разу було краще").
5. Провести тренування моделі (forward pass → backward pass → оновлення ваг).
6. Оцінити модель на тестовому наборі, обчислити точність та ймовірності дефолту.
- Мета:
  - побудувати модель, яка допоможе банку передбачати ризики кредитного дефолту.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

# Підготовка даних
np.random.seed(42)
data_size = 1_000

data = {
    'credit_amount': np.random.randint(1_000, 50_000, data_size),
    'income': np.random.randint(15_000, 150_000, data_size),
    'age': np.random.randint(18, 70, data_size),
    'loan_duration': np.random.randint(6, 60, data_size),
    'gender': np.random.choice(['male', 'female'], data_size),
    'marital_status': np.random.choice(['single', 'married', 'divorced'], data_size),
    'job_type': np.random.choice(['employed', 'self-employed', 'unemployed'], data_size),
    'default': np.random.randint(0, 2, data_size) # 0 = повернув, 1 = не повернув
}
df = pd.DataFrame(data)

# Створюємо формулу
df['default'] = ((df['credit_amount'] > 30_000) * 0.4 +
                 (df['income'] < 30_000) * 0.3 +
                 (df['age'] < 25) * 0.2 +
                 (df['loan_duration'] > 48) * 0.1 +
                 (df['gender'] == 'female') * 0.05 + 
                 (df['marital_status'] == 'single') * 0.05 +
                 (df['job_type'] == 'unemployed') * 0.1).apply(lambda x: 1 if x + np.random.rand() * 0.3 > 0.5 else 0)


print("Перші 5 рядків наших даних:")
print(df.head())
print("-" * 75)


print("\nІнформація про дані:")
df.info()
print("-" * 75)


numerical_features = ['credit_amount', 'income', 'age', 'loan_duration']
categorical_features = ['gender', 'marital_status', 'job_type']
target = 'default'

X = df.drop(columns=target)
y = df[target]

# Опрацюємо дані (масштабування та даммі-ознаки)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features), # Масштабування числових
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # Даммі для категоріальних
    ])

X_processed = preprocessor.fit_transform(X)  # Застосуємо трансформації

cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_feature_names)
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)
print(f"\nРозмір даних після опрацювання: {X_processed.shape}")
print("-" * 75)


# Розібємо дані на тренувальний та тестовий набори
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# Перетворюємо дані на тензори PyTorch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) 
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(f"\nРозмір тренувального X: {X_train_tensor.shape}, Y: {y_train_tensor.shape}")
print(f"Розмір тестового X: {X_test_tensor.shape}, Y: {y_test_tensor.shape}")
print("-" * 75)


# Створемо нейронну мережу
class CreditPredictor(nn.Module):
    def __init__(self, input_size):
        super(CreditPredictor, self).__init__()
        self.layer_1 = nn.Linear(input_size, 64) # Перший прихований шар
        self.relu_1 = nn.ReLU()
        self.layer_2 = nn.Linear(64, 32) # Другий прихований шар
        self.relu_2 = nn.ReLU()
        self.layer_3 = nn.Linear(32, 16) # Третій прихований шар
        self.relu_3 = nn.ReLU()
        self.output_layer = nn.Linear(16, 1) # Вихідний шар
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu_1(self.layer_1(x))
        x = self.relu_2(self.layer_2(x))
        x = self.relu_3(self.layer_3(x))
        x = self.sigmoid(self.output_layer(x))
        return x

input_size = X_train_tensor.shape[1]
model = CreditPredictor(input_size)
print("\nНаша нейронна мережа:")
print(model)
print("-" * 75)


# Визначемо функцію втрат та оптимізатор
criterion = nn.BCELoss()    # функція втрат 
optimizer = optim.Adam(model.parameters(), lr=0.001) # оптимізатор-вчитель


# Потренуємо модель
num_epochs = 100 # Кількість "епох" або проходів по всіх даних

print("\nПочинаємо тренування моделі...")
for epoch in range(num_epochs):
        # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
        # Backward pass та оптимізація
    optimizer.zero_grad() 
    loss.backward() 
    optimizer.step() 

    if (epoch + 1) % 10 == 0:
        print(f'Епоха [{epoch+1}/{num_epochs}], Втрати: {loss.item():.4f}')
print("Тренування завершено!")
print("-" * 75)


# Оцінюємо модель на тестовому наборі
model.eval() # Переводимо модель у режим оцінки
with torch.no_grad(): 
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)   
    predicted_classes = (test_outputs > 0.5).float()  # Перетворимо ймовірності в бінарні класи (0 або 1) 
    accuracy = accuracy_score(y_test_tensor.numpy(), predicted_classes.numpy())  # Обчислимо точності    
    probabilities_default = test_outputs.squeeze().numpy() # Обчислемо ймовірностей дефолту (вихід з Sigmoid)

    print(f'\nВтрати на тестовому наборі: {test_loss.item():.4f}')
    print(f'Точність на тестовому наборі: {accuracy:.4f}')
    print(f'\nПриклад передбачених ймовірностей дефолту для перших 10 тестових клієнтів:')
    print(probabilities_default[:10])
    print(f'\nПриклад передбачених класів (0/1) для перших 10 тестових клієнтів:')    
    print(predicted_classes[:10].squeeze().numpy())
    print(f'\nРеальні значення (0/1) для перших 10 тестових клієнтів:')
    print(y_test_tensor[:10].squeeze().numpy())
    print("-" * 75)

    
    # Добавимо додаткову метрику: AUC-ROC (Area Under the Receiver Operating Characteristic Curve)
    try:
        auc_roc = roc_auc_score(y_test_tensor.numpy(), probabilities_default)
        print(f'AUC-ROC на тестовому наборі: {auc_roc:.4f}')
    except ValueError:
        print("AUC-ROC не може бути обчислений, оскільки в тестовому наборі лише один клас.")

Перші 5 рядків наших даних:
   credit_amount  income  age  loan_duration  gender marital_status  \
0          16795  139040   39             31  female        married   
1           1860   19324   60             32  female         single   
2          39158   91323   32             13    male         single   
3          45732   24111   39             24    male        married   
4          12284   31389   54             47    male        married   

        job_type  default  
0  self-employed        0  
1       employed        1  
2  self-employed        1  
3     unemployed        1  
4       employed        0  
---------------------------------------------------------------------------

Інформація про дані:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   credit_amount   1000 non-null   int32 
 1   income          1000 non-null   int32

## Завдання 2: Передбачення відтоку користувачів мобільного додатку
### Опис:
    - Компанія, що розробляє мобільний додаток для фітнесу, хоче визначити, які користувачі можуть припинити користуватися додатком в найближчі місяці.
- Дані:
    - Числові ознаки:
        - sessions_per_week – середня кількість сесій на тиждень
        - avg_session_duration – середня тривалість сесії в хвилинах
        - workout_completed – середня кількість завершених тренувань
    - Категоріальні ознаки:
        - subscription_type – тип підписки (безкоштовна, преміум)
        - device_type – тип пристрою (iOS, Android)
    - Цільова змінна:
        - o churn (0 = залишився, 1 = відтік)
- Завдання студента:
1. Підготувати дані:
    - Перетворити категоріальні ознаки у числові.
    - Масштабувати числові ознаки.
2. Розділити дані на тренувальний та тестовий набори.
3. Побудувати нейронну мережу для бінарної класифікації:
    - 2 приховані шари з ReLU
    - Вихідний шар з Sigmoid
4. Вибрати функцію втрат та оптимізатор.
5. Навчити модель на тренувальних даних.
6. Оцінити точність та здатність моделі прогнозувати ймовірність відтоку на тестовому наборі.
- Мета:
      - визначити групу користувачів, які знаходяться у зоні ризику відтоку, щоб компанія могла запровадити цільові заходи утримання.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam


print("Крок 1: Створення зразкових даних (імітація реальних даних)")
data = {
    'sessions_per_week': [5, 3, 7, 2, 6, 4, 8, 1, 5, 7, 3, 6, 2, 4, 5, 8, 1, 7, 3, 6],
    'avg_session_duration': [30, 15, 45, 10, 35, 20, 50, 5, 25, 40, 18, 32, 12, 22, 28, 48, 8, 38, 16, 33],
    'workout_completed': [10, 3, 15, 2, 12, 5, 18, 1, 8, 14, 4, 11, 2, 6, 9, 17, 1, 13, 3, 10],
    'subscription_type': ['premium', 'free', 'premium', 'free', 'premium', 'free', 'premium', 'free', 'free', 'premium', 'free', 'premium', 'free', 'free', 'premium', 'premium', 'free', 'premium', 'free', 'premium'],
    'device_type': ['iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android', 'iOS', 'Android'],
    'churn': [0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0]
}
df = pd.DataFrame(data)
print("Перші 5 рядків наших даних:")
print("-" * 75)


print(df.head())
print("-" * 75)


# Визначимо числові та категоріальні ознаки
print("Крок 2: Підготовка даних - перетворення категоріальних та масштабування числових ознак.")
numeric_features = ['sessions_per_week', 'avg_session_duration', 'workout_completed']
categorical_features = ['subscription_type', 'device_type']

# Створимо "трансформер", який буде обробляти наші дані
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Розділяємо дані на ознаки (X) та цільову змінну (y)
X = df.drop('churn', axis=1)
y = df['churn']

# Розділемо дані на тренувальні та тестові набори 80% даних для навчання, 20% для тестування
print("Крок 3: Розділення даних на тренувальний (80%) та тестовий (20%) набори.")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Розмір тренувального набору: {X_train.shape[0]} зразків")
print(f"Розмір тестового набору: {X_test.shape[0]} зразків")
print("-" * 75)

# Застосуємо підготовку даних до наших наборів
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Визначимо кількість ознак після обробки для вхідного шару нейронної мережі
input_dim = X_train_processed.shape[1]
print(f"Кількість ознак після обробки: {input_dim}")
print("-" * 75)

# Побудуємо нейронну мережу для бінарної класифікації
print("Крок 4: Побудова нейронної мережі.")
model = Sequential([
    Input(shape=(input_dim,)),   
    Dense(32, activation='relu'),  # перший прихований шар
    Dense(16, activation='relu'),  # другий прихований шар
    Dense(1, activation='sigmoid') # вихідний шар для бінарної класифікації
])

# Виберемо функцію втрат та оптимізатор
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Структура нашої нейронної мережі:")
model.summary()
print("-" * 75)


# Навчаємо моделі на тренувальних даних
print("Крок 6: Навчання моделі...")
history = model.fit(X_train_processed, y_train,
                    epochs=50,  # кількість разів модель буде "проходити" по всіх даних
                    batch_size=4, # кількість зразків даних модель обробляє за один раз
                    verbose=1) # показує прогрес навчання
print("Модель навчена!")
print("-" * 75)


# Оцінемо моделі на тестовому наборі
print("Крок 7: Оцінка моделі на тестових даних.")
loss, accuracy = model.evaluate(X_test_processed, y_test, verbose=0)
print(f"Точність моделі на тестовому наборі: {accuracy:.4f}")
print("-" * 75)


y_pred_proba = model.predict(X_test_processed)  # Прогнозуємо ймовірності відтоку для тестового набору
y_pred = (y_pred_proba > 0.5).astype(int)       # Перетворюємо ймовірності в бінарні передбачення (0 або 1)
roc_auc = roc_auc_score(y_test, y_pred_proba)   # Розраховуємо якість бінарної класифікації
print(f"AUC-ROC на тестовому наборі: {roc_auc:.4f}")
print("-" * 75)


print("Крок 8: Визначення користувачів у зоні ризику відтоку.")
# Додамо передбачені ймовірності до оригінального тестового набору для аналізу
X_test_with_proba = X_test.copy()
X_test_with_proba['churn_probability'] = y_pred_proba
X_test_with_proba['predicted_churn'] = y_pred

# Сортуємо користувачів за ймовірністю відтоку (від найвищої до найнижчої)
risky_users = X_test_with_proba.sort_values(by='churn_probability', ascending=False)
print("\nКористувачі з найвищою ймовірністю відтоку (у тестовому наборі):")
print(risky_users.head())
print("-" * 75)


churn_threshold = 0.7
high_risk_users = risky_users[risky_users['churn_probability'] >= churn_threshold]
if not high_risk_users.empty:
    print(f"\nКористувачі у зоні високого ризику відтоку (ймовірність >= {churn_threshold}):")
    print(high_risk_users)
else:
    print(f"\nНе знайдено користувачів у зоні високого ризику відтоку (ймовірність >= {churn_threshold}) у тестовому наборі.")
    print("-" * 75)
print("\nЦі користувачі є цільовою групою для заходів утримання компанією.")

Крок 1: Створення зразкових даних (імітація реальних даних)
Перші 5 рядків наших даних:
---------------------------------------------------------------------------
   sessions_per_week  avg_session_duration  workout_completed  \
0                  5                    30                 10   
1                  3                    15                  3   
2                  7                    45                 15   
3                  2                    10                  2   
4                  6                    35                 12   

  subscription_type device_type  churn  
0           premium         iOS      0  
1              free     Android      1  
2           premium         iOS      0  
3              free     Android      1  
4           premium         iOS      0  
---------------------------------------------------------------------------
Крок 2: Підготовка даних - перетворення категоріальних та масштабування числових ознак.
Крок 3: Розділення даних на тренува

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 32)                  │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 801 (3.13 KB)

 Trainable params: 801 (3.13 KB)

 Non-trainable params: 0 (0.00 B)

---------------------------------------------------------------------------
Крок 6: Навчання моделі...
Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3750 - loss: 0.7507
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4375 - loss: 0.7164
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5625 - loss: 0.6878 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6250 - loss: 0.6644
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8125 - loss: 0.6446
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8125 - loss: 0.6286
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8125 - loss: 0.6145
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8125 - loss: 0.6000
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8125 - loss: 0.5848
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8125 - loss: 0.5709
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.87